<a href="https://colab.research.google.com/github/emrah1982/SmartFarmStrawberryDisease/blob/main/StrawberryVision_Colab_Production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍓 Çilek — YOLO26 Eğitimi (Colab Pro+)

Bu notebook **hiyerarşik çok modelli** mimari için model eğitir. Tek büyük model
yerine her problem alanı için **bağımsız model** kullanılır:

```
görüntü → organ modeli (Leaf/Fruit/Flower)
        → ROI kırpma
        → o organda tetiklenen uzman model
        → sonuç
```

**Neden?** Tek modelde sınıflar birbirine karışıyordu: olgunluk sınıfları
**yaprakları** "olgunlaşmamış çilek" sanıyordu ve iki grubun güven aralıkları üst
üste bindiği için ayıran eşik yoktu. Hiyerarşide olgunluk modeli yalnızca **meyve
kırpıntısı** görür — hata yapısal olarak imkânsızlaşır.
Ayrıntı: `docs/MIMARI_GECIS_PLANI.md`

## 1️⃣ Hangi modeli eğiteceksiniz?

**4️⃣ hücresindeki** tek değişken belirler:

```python
EGITILECEK = 'organ_detection'
```

| Değer | Ne eğitir | train / valid / test | Sınıf | Paket |
|-------|-----------|----------------------|-------|-------|
| `organ_detection` ⭐ | Leaf · Fruit · Flower | 14.313 / 1.363 / 682 | 3 | 838 MB |
| `leaf_disease` | Yaprak hastalıkları | 5.641 / 1.169 / 426 | 4 | 211 MB |
| `fruit_disease` | Meyve hastalıkları | 4.662 / 658 / 228 | 4 | 182 MB |
| `fruit_ripeness` | Olgunluk | 3.723 / 468 / 100 | 3 | 243 MB |
| `bocek_teshis` | Böcek teşhisi (ayrı akış) | 3.109 / 565 / 225 | 6 | 118 MB |
| `pest_detection` | Saha zararlıları | — | — | *veri toplanacak* |
| `birlesik` | Eski tek model (10 sınıf) | 10.780 / 1.793 / 570 | 10 | 772 MB |

⭐ **Önce `organ_detection`** — o olmadan hiyerarşi çalışmaz, sistem tek modele düşer.
Diğerlerinin sırası önemsizdir, paralel de eğitilebilir.

## 2️⃣ Çalıştırmadan önce — 2 zorunlu adım

**a) Runtime:** Runtime → Change runtime type →
- Hardware accelerator: **A100 GPU** ⭐ (yoksa L4)
- Runtime shape: **High-RAM**
- Runtime → **Background execution** açık olsun (tarayıcı kapansa da eğitim sürer)

**b) Dataset'i Drive'a yükleyin (model başına bir kez):**

Paketleri yerelde üretin:
```bash
python scripts/dataset_ayir.py --paketle      # datasets/cilek/*.zip
```

Sonra `datasets/cilek/` klasörünü Drive'a **olduğu gibi** kopyalayın:
```
MyDrive/SmartFarmStrawberryDisease/datasets/cilek/organ_detection.zip
                                   └─ depodaki düzenin aynısı ─┘
```
Yalnızca `.zip` dosyaları yeterli — açılmış klasörleri yüklemeyin, hem yer
kaplar hem Drive üzerinden okumak yavaştır (Colab zip'i yerel diske alıp açar).

> Eski `dataset/<ad>.zip` konumu da çalışır; bulunamazsa tüm Drive taranır.

Sonra **Runtime → Run all**. Tek elle müdahale: Drive bağlama hücresi bir kez yetki
onayı ister.

## 3️⃣ Eğitim bitince

Çıktılar **Drive'a** yazılır (oturum kapansa da kalır):

| Ne | Nerede |
|----|--------|
| Koşu dizini, checkpoint'ler | `results/<koşu>/` |
| Boru hattı adıyla model | `best_models/cilek/organ.pt` |

Bu dosyayı indirip projede `models/cilek/` altına koyun; doğrulayarak kurmak için:

```bash
python scripts/model_kur.py organ <indirdiginiz.pt>
```

Model doğru adla yerine konduğu anda **beş analiz yolu birden** (fotoğraf, ayrıntılı,
video, IP kamera, canlı akış) onu kullanmaya başlar — kod değişikliği gerekmez.

> ⚠️ **Sekme eski kalabilir.** Colab, GitHub'dan açılan notebook'u tarayıcıda
> önbelleğe alır; `git pull` scriptleri günceller ama **hücre kodunu güncellemez**.
> 3️⃣ hücresi sürüm farkını fark edip uyarır. Uyarı görürseniz: sekmeyi kapatın,
> linki tekrar açın, **Ctrl+Shift+R**.

## 1️⃣ Paket kurulumu ve uyumluluk kontrolü

In [1]:
# Colab'da SADECE ultralytics kurulur.
# NEDEN: Colab'da torch / numpy / opencv zaten kurulu ve birbiriyle uyumludur.
# Bunları elle kurmak veya yükseltmek ikili (ABI) uyumsuzluğu yaratır
# ("numpy.dtype size changed", "cv2 import error") ve runtime restart gerektirir.
# ultralytics eksik bağımlılıklarını uyumlu sürümlerle kendisi çeker.
!pip install -q "ultralytics>=8.3.200"

print('\n--- Sürüm / donanım kontrolü ---')
problem = False
try:
    import numpy, torch, cv2, ultralytics, psutil
    print('numpy      :', numpy.__version__)
    print('torch      :', torch.__version__)
    print('opencv     :', cv2.__version__)
    print('ultralytics:', ultralytics.__version__)

    v = tuple(int(x) for x in ultralytics.__version__.split('.')[:3])
    if v < (8, 3, 200):
        print('\n⚠️ ultralytics sürümü YOLO26 için eski: !pip install -U ultralytics')
        problem = True

    ram = psutil.virtual_memory().total / 1e9
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'\n✅ GPU: {name} ({vram:.0f} GB VRAM) | RAM: {ram:.0f} GB')

        up = name.upper()
        if 'A100' in up:
            print('🏆 En hızlı seçenek aktif.')
        elif 'L4' in up:
            print('👍 L4 iyi bir seçim. Daha hızlısı için: Runtime > Change runtime type > A100 GPU')
        else:
            print(f'ℹ️ Pro+ ile daha hızlısı mümkün: Runtime > Change runtime type > A100 GPU')

        if ram < 60:
            print('💡 Runtime shape: High-RAM seçerseniz A100\'de cache=ram açılır ve eğitim hızlanır.')
    else:
        print('\n⚠️ GPU YOK! Runtime > Change runtime type > A100 GPU seçin, sonra bu hücreyi tekrar çalıştırın.')
        problem = True
except Exception as e:
    print('\n❌ Import hatası:', e)
    print('💡 Çözüm: Runtime > Restart session, sonra bu hücreyi TEKRAR çalıştırın.')
    problem = True

print('\n' + ('⚠️ Yukarıdaki uyarıyı giderin' if problem else '✅ Ortam hazır'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.2 MB/s eta 0:00:00

--- Sürüm / donanım kontrolü ---
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
numpy      : 2.0.2
torch      : 2.11.0+cu128
opencv     : 4.13.0
ultralytics: 8.4.106

✅ GPU: Tesla T4 (16 GB VRAM) | RAM: 14 GB
ℹ️ Pro+ ile daha hızlısı mümkün: Runtime > Change runtime type > A100 GPU
💡 Runtime shape: High-RAM seçerseniz A100'de cache=ram açılır ve eğitim hızlanır.

✅ Ortam hazır


## 2️⃣ Google Drive bağlantısı

Bu hücre bir kez **yetki onayı** ister (açılan pencereden hesabınızı seçin).

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/SmartFarmStrawberryDisease')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
RESULTS_DIR = DRIVE_ROOT / 'results'
MODELS_DIR = DRIVE_ROOT / 'best_models'
for d in (DRIVE_ROOT, CHECKPOINT_DIR, RESULTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('✅ Drive hazır:', DRIVE_ROOT)

## 3️⃣ Depoyu indir / güncelle

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/emrah1982/SmartFarmStrawberryDisease.git'
REPO_DIR = Path('/content/SmartFarmStrawberryDisease')

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=False)
else:
    os.chdir('/content')
    subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir(REPO_DIR)

print('CWD:', Path.cwd())
# Yapılandırma ürün kapsamlıdır: configs/urunler/<urun>/veri.yaml
# (çok bitkili kurulum, bkz. docs/COK_BITKILI_YAPI.md). Eski kurulumlarda
# kapsamsız configs/strawberry_data.yaml vardı; ikisi de kabul edilir.
_urun = globals().get("URUN", "cilek")
_adaylar = [Path.cwd() / "configs" / "urunler" / _urun / "veri.yaml",
            Path.cwd() / "configs" / "strawberry_data.yaml"]
_bulunan = next((y for y in _adaylar if y.exists()), None)
if _bulunan is None:
    _liste = chr(10).join("  - " + str(y) for y in _adaylar)
    raise AssertionError(
        "Eğitim yapılandırması bulunamadı. Aranan yollar:" + chr(10) + _liste +
        chr(10) + "Depo doğru klonlanmamış veya güncel değil olabilir." +
        chr(10) + "Çözüm: hücreyi tekrar çalıştırın (git pull yapar).")
print("✅ Depo hazır —", _bulunan.relative_to(Path.cwd()))

# ── Sekme bayat mı? ────────────────────────────────────────────────────────
# Colab, GitHub'dan açılan notebook'u TARAYICIDA önbelleğe alır. Depo git pull
# ile güncellenir ama SEKMEDEKİ HÜCRE KODU eski kalır. Bu fark defalarca
# "dosya yok" tipi hatalara yol açtı: script güncel, hücre eski.
# Aşağıdaki karşılaştırma bunu sessiz bırakmaz.
NOTEBOOK_SURUM = "2026-07-30-1"

try:
    _nb = Path.cwd() / "StrawberryVision_Colab_Production.ipynb"
    _depo_surum = ""
    if _nb.exists():
        import json as _json
        for _c in _json.loads(_nb.read_text(encoding="utf-8"))["cells"]:
            _s = "".join(_c["source"])
            if "NOTEBOOK_SURUM = " in _s:
                _depo_surum = _s.split('NOTEBOOK_SURUM = "')[1].split('"')[0]
                break
    if _depo_surum and _depo_surum != NOTEBOOK_SURUM:
        print("")
        print("=" * 74)
        print("⚠️  BU SEKMEDEKİ NOTEBOOK ESKİ")
        print("=" * 74)
        print(f"   sekmedeki sürüm : {NOTEBOOK_SURUM}")
        print(f"   depodaki sürüm  : {_depo_surum}")
        print("")
        print("   Scriptler git pull ile güncellendi ama HÜCRE KODU eski kaldı;")
        print("   tutarsız davranış görebilirsiniz (ör. taşınmış dosya yolları).")
        print("")
        print("   Çözüm: sekmeyi KAPATIN, linki tekrar açın ve Ctrl+Shift+R yapın.")
        print("=" * 74)
except Exception:
    pass                      # sürüm kontrolü asla eğitimi engellemesin


## 4️⃣ Dataset — model seçimi, Drive'dan açma ve doğrulama

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  BU HÜCRE, EĞİTİMİN TAMAMINI BELİRLER — dataset de buna göre indirilir.
# ═══════════════════════════════════════════════════════════════════════════
EGITILECEK = 'organ_detection'
URUN       = 'cilek'
# ═══════════════════════════════════════════════════════════════════════════
#   'organ_detection' ⭐ Leaf/Fruit/Flower   14.313/1.363/682   3 sınıf   838 MB
#   'leaf_disease'      yaprak hastalıkları   5.641/1.169/426   4 sınıf   211 MB
#   'fruit_disease'     meyve hastalıkları    4.662/  658/228   4 sınıf   182 MB
#   'fruit_ripeness'    olgunluk              3.723/  468/100   3 sınıf   243 MB
#   'bocek_teshis'      böcek teşhisi (AYRI)  3.109/  565/225   6 sinif   118 MB
#   'pest_detection'    saha zararlıları      — (veri toplanacak)
#   'birlesik'          eski tek model       10.780/1.793/570  10 sınıf   772 MB
#
#   ⭐ Önce organ_detection: o olmadan hiyerarşi çalışmaz, sistem tek modele düşer.
#      Diğerlerinin sırası önemsizdir. Ayrıntı: docs/MIMARI_GECIS_PLANI.md
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path

if EGITILECEK == 'birlesik':
    # Yapılandırma ürünün klasöründedir; eski kurulumlarda kapsamsız yola düşer.
    _yeni = REPO_DIR / 'configs' / 'urunler' / URUN / 'veri.yaml'
    _eski = REPO_DIR / 'configs' / 'strawberry_data.yaml'
    DATA_YAML_PATH = str((_yeni if _yeni.exists() else _eski).resolve())
    ARSIV_ADI = 'dataset_colab.zip'
else:
    # Uzman dataset'ler `python scripts/dataset_ayir.py --paketle` ile üretilir
    # ve <ad>.zip olarak Drive'daki dataset/ klasörüne yüklenir.
    DATA_YAML_PATH = str((REPO_DIR / 'datasets' / URUN / EGITILECEK / 'data.yaml').resolve())
    ARSIV_ADI = f'{EGITILECEK}.zip'

print(f'🎯 Eğitilecek model : {EGITILECEK}   (ürün: {URUN})')
print(f'📦 Gereken arşiv    : {ARSIV_ADI}')
print(f'📁 Dataset config   : {DATA_YAML_PATH}')
print()
print("   Bu arşiv Drive'da şurada olmalı:")
if EGITILECEK == 'birlesik':
    print(f'   {DRIVE_ROOT / "dataset" / ARSIV_ADI}')
else:
    # Uzman paketler depodaki datasets/<urun>/ duzenini aynen korur; Drive'a da
    # oyle yuklemek en az karisan yoldur. Eski dataset/ klasoru de desteklenir.
    print(f'   {DRIVE_ROOT / "datasets" / URUN / ARSIV_ADI}')

### 🔍 Drive kontrolü — arşiv doğru klasörde mi?
Hiçbir şeyi değiştirmez; yalnızca Drive'da ne olduğunu gösterir.

In [ ]:
# 🔍 DRIVE TARAYICI — arşiv doğru klasörde mi?
# Bu hücre hiçbir şeyi değiştirmez; sadece Drive'da NE olduğunu gösterir.
import os
from pathlib import Path

MYDRIVE = Path('/content/drive/MyDrive')
ZIP_NAME = ARSIV_ADI                      # yukarıdaki seçim hücresinden gelir

# Aranacak klasörler, hazırlık scriptiyle AYNI sırada olmalı; yoksa tarayıcı
# "yok" derken script buluyor (ya da tersi) ve kimse nereye bakacağını bilmiyor.
if EGITILECEK == 'birlesik':
    ARANAN = [DRIVE_ROOT / 'dataset', DRIVE_ROOT]
else:
    ARANAN = [DRIVE_ROOT / 'datasets' / URUN, DRIVE_ROOT / 'datasets',
              DRIVE_ROOT / 'dataset' / URUN, DRIVE_ROOT / 'dataset', DRIVE_ROOT]


def listele(d, baslik, limit=25):
    print(f'\n📂 {baslik}')
    print(f'   {d}')
    if not d.exists():
        print('   ❌ BU KLASÖR YOK')
        return
    items = sorted(d.iterdir(), key=lambda q: (not q.is_dir(), q.name.lower()))
    if not items:
        print('   (boş)')
    for q in items[:limit]:
        if q.is_dir():
            print(f'   📁 {q.name}/')
        else:
            try:
                print(f'   📄 {q.name}   ({q.stat().st_size/1e6:.1f} MB)')
            except OSError:
                # Drive'da kisayol/cop kutusu girdileri listelenir ama acilamaz
                print(f'   📄 {q.name}   (boyut okunamadi — kisayol olabilir)')
    if len(items) > limit:
        print(f'   ... +{len(items)-limit} öğe daha')


print('=' * 62)
print(f'ARANAN ARŞİV: {ZIP_NAME}')
print('=' * 62)
listele(DRIVE_ROOT, 'Proje klasörü (DRIVE_ROOT)')
for d in ARANAN[:2]:
    listele(d, 'Aday klasör')

print('\n' + '=' * 62)
print('ARŞİV ARAMASI')
print('=' * 62)
bulundu = None
for d in ARANAN:
    aday = d / ZIP_NAME
    var = aday.exists()
    print(f'   {"✅" if var else "  "} {aday}')
    if var and bulundu is None:
        bulundu = aday

# MyDrive'da 4 seviyeye kadar tüm zip'leri listele (yanlış klasöre yüklendiyse görünür)
bulunan = []
for cur, dirs, files in os.walk(MYDRIVE):
    if len(Path(cur).relative_to(MYDRIVE).parts) >= 4:
        dirs[:] = []
        continue
    for f in files:
        if f.lower().endswith('.zip'):
            q = Path(cur) / f
            try:
                bulunan.append((q, q.stat().st_size / 1e6))
            except OSError:
                continue   # Drive kisayolu/erisilemeyen girdi — atla

if bulunan:
    print("\nDrive'daki .zip dosyaları:")
    for q, mb in sorted(bulunan, key=lambda x: -x[1]):
        isaret = '  ⬅ BU KULLANILACAK' if q == bulundu else ''
        print(f'   {mb:>7.1f} MB   {q}{isaret}')
else:
    print("\n⚠️ Drive'da (4 seviyeye kadar) hiç .zip bulunamadı.")

print('\n' + '=' * 62)
if bulundu:
    print(f'✅ {ZIP_NAME} bulundu → sonraki hücreyi çalıştırabilirsiniz.')
elif any(q.name == ZIP_NAME for q, _ in bulunan):
    yer = next(q for q, _ in bulunan if q.name == ZIP_NAME)
    print(f'⚠️ {ZIP_NAME} Drive\'da var ama beklenmedik yerde:')
    print(f'   {yer}')
    print('   Hazırlık hücresi yine de bulur (tüm Drive taranır), sadece yavaş olur.')
else:
    print(f'❌ {ZIP_NAME} yok. Yerelde üretip yükleyin:')
    print('   python scripts/dataset_ayir.py --paketle')
    print(f'   → datasets/{URUN}/{ZIP_NAME} dosyasını şuraya yükleyin:')
    print(f'   {ARANAN[0]}')


In [ ]:
# Dataset hazırlığı DEPODAKİ scripte devredildi.
# NEDEN: Colab sekmesi açıkken hücre kodu önbellekte kalır; dosyayı güncellesek
# bile eski kod çalışır. Script depoda olduğu için yukarıdaki "git pull" adımı
# her çalıştırmada en güncel halini indirir — düzeltmeler anında yansır.
import subprocess, sys
from pathlib import Path

DATASET_DIR = REPO_DIR / 'dataset'

sonuc = subprocess.run(
    [sys.executable, 'scripts/prepare_colab_dataset.py',
     '--drive-root', str(DRIVE_ROOT), '--repo', str(REPO_DIR),
     '--model', EGITILECEK, '--urun', URUN],
    cwd=str(REPO_DIR),
)
if sonuc.returncode != 0:
    raise RuntimeError('Dataset hazırlanamadı — yukarıdaki mesaja bakın.')


In [ ]:
# Eğitimden ÖNCE doğrulama: her dizin var mı, görüntü/label eşleşiyor mu?
import yaml
from pathlib import Path

# MUTLAK YOL ZORUNLU: Ultralytics dataset kökünü data.yaml'ın bulunduğu dizinden türetir.
# Göreli yol verilirse kökü kendi DATASETS_DIR'i altında arar → "images not found".
# REPO_DIR üzerinden kurulur ki çalışma dizini değişse bile doğru kalsın.
cfg = yaml.safe_load(Path(DATA_YAML_PATH).read_text(encoding='utf-8'))
root = Path(cfg.get('path') or Path(DATA_YAML_PATH).parent)
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

names = cfg['names']
print('📁 Config:', DATA_YAML_PATH)
print('🏷️  Sınıflar:', cfg['nc'], '→', list(names.values()) if isinstance(names, dict) else names)
print()

ok = True
for split in ('train', 'val', 'test'):
    entries = cfg.get(split) or []
    entries = [entries] if isinstance(entries, str) else entries
    n_img = n_lbl = n_bg = 0
    for e in entries:
        d = (root / e).resolve()
        if not d.exists():
            print(f'❌ {split}: dizin YOK → {d}'); ok = False; continue
        ld = Path(str(d).replace('/images', '/labels'))
        if not ld.exists():
            print(f'❌ {split}: labels dizini YOK → {ld}'); ok = False; continue
        imgs = [p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS]
        missing = sum(1 for p in imgs if not (ld / f'{p.stem}.txt').exists())
        n_bg += sum(1 for p in imgs if (ld / f'{p.stem}.txt').exists()
                    and (ld / f'{p.stem}.txt').stat().st_size == 0)
        if missing:
            print(f"⚠️ {split}: {missing} görüntünün label'ı yok → {d.parent.parent.name}"); ok = False
        n_img += len(imgs); n_lbl += len(imgs) - missing
    print(f'{split:<6}: {len(entries)} dizin | {n_img:>5} görüntü | {n_lbl:>5} label | {n_bg} background')

print('\n' + ('✅ Dataset eğitime hazır' if ok else '❌ Sorun var — yukarıdaki uyarılara bakın'))

## 5️⃣ Eğitim konfigürasyonu (GPU'ya göre otomatik optimize)

Parametreler ve gerekçeleri `configs/train_config.yaml` içindeki yorumlardadır.
Aşağıdaki hücre **çalışan GPU'yu algılayıp** `batch` / `workers` / `cache` değerlerini
otomatik ayarlar — elle bir şey değiştirmenize gerek yoktur.

| GPU | VRAM | batch (imgsz 1024) | 200 epoch tahmini |
|---|---|---|---|
| **A100** ⭐ | 40 GB | 32 | ~3-4 saat |
| L4 | 24 GB | 16 | ~7-9 saat |
| V100 | 16 GB | 8 | ~10-12 saat |
| T4 | 16 GB | 8 | 15+ saat |

> ⚠️ **GPU tipi koddan seçilemez** — Colab'ın runtime ayarıdır.
> Pro+ ile: **Runtime → Change runtime type → A100 GPU** + **High-RAM** seçin.
> High-RAM açıkken A100 profilinde `cache='ram'` devreye girer: görüntüler belleğe
> alınır, disk okuma darboğazı kalkar ve epoch süresi belirgin düşer.

> 💡 **Pro+ avantajı:** Runtime menüsünden **Background execution**'ı açarsanız
> tarayıcıyı kapatsanız bile eğitim sürer — 200 epoch'u tek oturumda bitirebilirsiniz.

In [ ]:
import yaml, torch, psutil
from pathlib import Path

TRAIN_CONFIG = yaml.safe_load((REPO_DIR / 'configs' / 'train_config.yaml').read_text(encoding='utf-8'))

# --- GPU'ya göre OTOMATİK optimizasyon --------------------------------------
# Colab Pro+ farklı GPU'lar verebilir (A100 40GB / L4 24GB / V100 16GB / T4 16GB).
# batch, VRAM ile doğru orantılı seçilir: çok büyük → CUDA OOM, çok küçük →
# GPU boşta bekler ve eğitim gereksiz uzar. Aşağıdaki değerler yolo26s +
# imgsz 1024 için güvenli üst sınırlardır (~%80 VRAM kullanımı).
PROFILES = {           # (batch @ imgsz1024, workers, açıklama)
    'A100': (32, 12, 'en hızlı — 200 epoch ~3-4 saat'),
    'L4':   (16,  8, 'iyi denge — 200 epoch ~7-9 saat'),
    'V100': ( 8,  8, 'orta — 200 epoch ~10-12 saat'),
    'T4':   ( 8,  8, 'yavaş — 200 epoch 15+ saat, imgsz 640 önerilir'),
}

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
key = next((k for k in PROFILES if k in gpu_name.upper().replace(' ', '')), None)

if key:
    batch, workers, note = PROFILES[key]
    TRAIN_CONFIG['batch'] = batch
    TRAIN_CONFIG['workers'] = workers
    print(f'🎯 GPU: {gpu_name} ({vram:.0f} GB) → {key} profili')
    print(f'   batch={batch}, workers={workers}  ({note})')
else:
    TRAIN_CONFIG['batch'] = max(4, int(vram // 1.3)) if vram else 4
    print(f'ℹ️ Tanınmayan GPU: {gpu_name} ({vram:.0f} GB) → batch={TRAIN_CONFIG["batch"]} (tahmini)')

TRAIN_CONFIG['amp'] = True   # karma hassasiyet; A100'de otomatik bf16

# --- RAM önbelleği: VERİ SETİ BÜYÜKLÜĞÜNE göre karar ------------------------
# Görüntüleri RAM'e almak disk darboğazını kaldırır ama gereken bellek görüntü
# SAYISI ile doğru orantılıdır: her görüntü imgsz x imgsz x 3 bayt olarak
# AÇILMIŞ halde tutulur. 13.000 görüntü @1024 => ~41 GB.
#
# Sadece "A100 + High-RAM" bakmak yetmez: veri seti büyüdükçe aynı makinede
# sınır aşılır ve oturum RAM yetersizliğinden ÖLDÜRÜLÜR. Belirtisi tipiktir:
# eğitim 1-2 epoch ilerler, sonra HATA VERMEDEN durur. Ayrıca her dataloader
# worker'ı bu belleğin bir kısmını kopyalar (Python refcount, copy-on-write'ı
# bozar) — bu yüzden önbellek açıkken worker sayısı da sınırlanır.
ram_gb = psutil.virtual_memory().total / 1e9
_imgsz = int(TRAIN_CONFIG.get('imgsz', 1024))
# Sayim SECILEN modelin dataset'inde yapilir. Sabit `dataset/` yazilirsa
# uzman modellerde 0 gorsel sayilir, onbellek sessizce kapanir ve "profil uygun
# degil" gibi alakasiz bir gerekce basilir — hizli A100'de bile disk darbogazi.
if EGITILECEK == 'birlesik':
    _ds = REPO_DIR / 'dataset'
else:
    _ds = Path(DATA_YAML_PATH).parent
try:
    _n_gorsel = sum(1 for f in _ds.rglob('*')
                    if f.parent.name == 'images'
                    and f.suffix.lower() in ('.jpg', '.jpeg', '.png'))
except Exception:
    _n_gorsel = 0
print(f'   dataset: {_ds}  ({_n_gorsel} görüntü)')

_tahmin_gb = _n_gorsel * _imgsz * _imgsz * 3 / 1e9
_sinir_gb = ram_gb * 0.40          # kalan: worker kopyaları + torch + OS

if key == 'A100' and ram_gb > 60 and 0 < _tahmin_gb < _sinir_gb:
    TRAIN_CONFIG['cache'] = 'ram'
    TRAIN_CONFIG['workers'] = min(int(TRAIN_CONFIG.get('workers', 8)), 8)
    print(f'   cache=ram — {_n_gorsel} görüntü ~{_tahmin_gb:.0f} GB '
          f'(sınır {_sinir_gb:.0f} GB, RAM {ram_gb:.0f} GB), workers={TRAIN_CONFIG["workers"]}')
else:
    TRAIN_CONFIG['cache'] = False
    if _tahmin_gb >= _sinir_gb:
        print(f'   cache=False — {_n_gorsel} görüntü @{_imgsz} ~{_tahmin_gb:.0f} GB RAM '
              f'isterdi, sınır {_sinir_gb:.0f} GB (RAM {ram_gb:.0f} GB)')
        print('      Önbellek açılsaydı oturum epoch 1-2 civarında sessizce ölürdü.')
        print(f'      Hız istiyorsanız: imgsz 640 (~{_n_gorsel * 640 * 640 * 3 / 1e9:.0f} GB) '
              "veya OVERRIDES = {'cache': 'disk'} (RAM yerine diske yazar)")
    elif _n_gorsel == 0:
        print(f'   cache=False — {_ds} içinde görüntü sayılamadı.')
        print('      Dataset henüz açılmamış olabilir; 4️⃣ hücresini çalıştırın.')
    else:
        print(f'   cache=False (GPU/RAM profili uygun değil — {key or "?"}, RAM {ram_gb:.0f} GB)')

# --- Epoch sayısı: geçmiş eğitimlerin eğrisinden ÖLÇÜLÜR -------------------
# "Büyük verelim, ezberlerse erken durdurma keser" yaklaşımı burada yanıltıcıdır:
# öğrenme oranı takvimi ve close_mosaic TOPLAM epoch'a göre hesaplanır. 200 verip
# 70'te durursanız model lr0'ın ~%73'ünde, takvimin ortasında kalır.
# Bu yüzden epoch, geçmiş koşuların doyma noktasına göre seçilir.
try:
    import sys as _sys
    _sys.path.insert(0, str(REPO_DIR / 'scripts'))
    import epoch_oner as _eo
    import importlib as _il
    _il.reload(_eo)
    # Hangi kosular sayilir? Doyma noktasi MODELE ve DATASET'e ozgudur:
    # 10 sinifli birlesik kosunun egrisi, 3 sinifli organ modeli icin gecerli
    # degildir. Once AYNI modelin kosulari aranir; yoksa digerleri yalnizca
    # kaba bir on bilgi olarak, acikca uyarilarak kullanilir.
    _hepsi = []
    for _csv in sorted(Path(RESULTS_DIR).rglob('results.csv')):
        _egri = _eo.egri_oku(_csv)
        if len(_egri) >= 5:
            _hepsi.append((_csv.parent.name, _eo.coz(_egri)))

    _ad = globals().get('EGITILECEK', 'birlesik')
    _kendi = [t for t in _hepsi if t[0].startswith(_ad)]
    _ayni_model = bool(_kendi)
    _analiz = _kendi or _hepsi

    if _hepsi and not _ayni_model:
        print(f'⚠️ {_ad} için geçmiş koşu YOK. Aşağıdakiler BAŞKA modellerin')
        print('   koşularıdır (farklı sınıf sayısı, farklı dataset) — doyma')
        print('   noktaları bu modele birebir uymaz, kaba bir fikir verir.')
        print()

    for _ad_k, _s in _analiz:
        _eo.rapor(_ad_k, _s)
    if _analiz:
        _ONERI = _eo.oner(_analiz, ince_ayar=(globals().get('MOD') == 'ince_ayar'))
        print(chr(10) + 'ℹ️ Yapılandırmadaki değer kullanılıyor: '
              f'epochs={TRAIN_CONFIG["epochs"]}, patience={TRAIN_CONFIG.get("patience")}')
        print('   Öneriyi uygulamak isterseniz: '
              "OVERRIDES = {'epochs': %d, 'patience': %d}" % (_ONERI['epochs'],
                                                              _ONERI['patience']))
except Exception as _e:
    print(f'ℹ️ Epoch önerisi hesaplanamadı ({type(_e).__name__}) — yapılandırma kullanılacak.')

# --- Elle geçersiz kılma (isteğe bağlı) -------------------------------------
# Hızlı ilk tur:    {'epochs': 50, 'imgsz': 640, 'batch': 32}
# OOM alırsanız:    {'batch': <yarısı>}
OVERRIDES = {}
TRAIN_CONFIG.update(OVERRIDES)

# Sonuçlar doğrudan Drive'a yazılsın (oturum kopsa bile checkpoint'ler kalır)
TRAIN_CONFIG['project'] = str(RESULTS_DIR)

# Kosu adi secilen modelden gelir: farkli modeller ayni klasoru paylasirsa
# find_run_dir() baskasinin best.pt'sini okur ve yanlis modeli degerlendiririz.
if EGITILECEK not in ('birlesik', ''):
    TRAIN_CONFIG['name'] = EGITILECEK


def find_run_dir():
    """Gerçek koşu dizinini bulur.

    exist_ok=False olduğu için Ultralytics her yeni eğitimde strawberry_exp2,
    strawberry_exp3 ... oluşturur. Sabit ismi varsayarsak ESKİ koşunun
    sonuçlarını okur ve yanlış modeli değerlendiririz — bu yüzden en son
    değiştirilen dizin seçilir.
    """
    base = Path(TRAIN_CONFIG['project'])
    cands = [p for p in base.glob(TRAIN_CONFIG['name'] + '*') if p.is_dir()]
    return max(cands, key=lambda p: p.stat().st_mtime) if cands else base / TRAIN_CONFIG['name']



def dataset_hazirla():
    """dataset/ hazır değilse depodaki hazırlık scriptini çalıştırır.

    Eğitim ve değerlendirme hücreleri bunu kendisi çağırır: 4️⃣ hücresini
    atlasanız veya oturum yenilense bile dataset otomatik hazırlanır.
    Zaten hazırsa saniyeler içinde döner, yeniden açma yapmaz.
    """
    import subprocess, sys
    r = subprocess.run(
        [sys.executable, 'scripts/prepare_colab_dataset.py',
         '--drive-root', str(DRIVE_ROOT), '--repo', str(REPO_DIR),
         '--model', globals().get('EGITILECEK', 'birlesik'),
         '--urun', globals().get('URUN', 'cilek')],
        cwd=str(REPO_DIR),
    )
    if r.returncode != 0:
        raise RuntimeError('Dataset hazırlanamadı — yukarıdaki mesaja bakın.')


print('\n--- Eğitim ayarları ---')
for k in ('model', 'epochs', 'batch', 'imgsz', 'workers', 'optimizer',
          'cos_lr', 'amp', 'cache', 'patience', 'save_period'):
    print(f'  {k}: {TRAIN_CONFIG.get(k)}')
print('\n  sonuç dizini:', TRAIN_CONFIG['project'])

## 6️⃣ Eğitim

Hücrenin başındaki **`MOD`** ayarı ne yapılacağını belirler:

| MOD | Ne yapar |
|---|---|
| `'otomatik'` ⭐ | Yarım kalmış eğitim varsa **devam eder**, bitmişse **atlar**, hiç yoksa başlatır |
| `'devam'` | Yarım kalmış eğitimden **devam etmeye zorlar**; yoksa hata verir (sessizce sıfırdan başlamaz) |
| `'sifirdan'` | Mevcut checkpoint'leri **yok sayar**, yepyeni koşu açar |
| `'ince_ayar'` 🔁 | **Mevcut modelinizden devam eder** (warm start): ağırlıklar devralınır, sıfırdan başlatılmaz. Yarım kalmış bir ince ayar varsa **kaldığı yerden sürdürür** |

> 🔁 **İnce ayar yarım kalırsa ne olur?** Hücre yeniden çalıştırıldığında yarım kalmış
> ince ayar koşusunu bulup **kaldığı yerden devam eder** — Ultralytics optimizer durumunu
> ve öğrenme oranı takvimini checkpoint'ten okur, baştan başlamaz. Sıfırdan koşular ve
> ince ayar koşuları ayrı değerlendirilir: biri diğerinin üstüne yazmaz, ve her koşu
> **kendi hedef epoch sayısıyla** kıyaslanır (60 epochluk ince ayar, 200 hedefli sıfırdan
> eğitimle karıştırılıp "yarım" sanılmaz).
>
> Baştan başlatmak isterseniz o koşu klasörünü silin.

> 🔁 **`'ince_ayar'` ile `'devam'` aynı şey değildir.**
> `'devam'` yarım kalmış bir koşuyu sürdürür ve ayarları checkpoint'ten okur —
> yeni veriyi/ayarı almaz. `'ince_ayar'` ise **yeni bir koşu** açar ve yalnızca
> başlangıç ağırlıklarını mevcut modelinizden alır; veri setine yeni görüntü
> eklediğinizde kullanılan budur.
>
> İnce ayarda **sınıf listesi kontrol edilir**: başlangıç ağırlığının sınıfları
> dataset ile birebir aynı değilse eğitim **başlatılmaz** ve sebebi yazılır.
> Sıfırdan eğitim ayarlarıyla ince ayar yapmayın — `configs/finetune_config.yaml`
> kullanın (epochs 70, AdamW, lr0 0.0008).

> ⚠️ **Devam ederken `OVERRIDES` etkisizdir** — Ultralytics ayarları checkpoint'ten
> okur. Ayar değiştirip yeniden eğitmek istiyorsanız `MOD = 'sifirdan'` kullanın.

Hücre çalışınca önce **mevcut koşuların listesini** basar: hangi eğitim kaçıncı
epoch'ta, tamamlanmış mı yarım mı — hepsi görünür.

Her `save_period` epoch'ta checkpoint doğrudan Drive'a yazılır; oturum koparsa
hücreyi tekrar çalıştırmanız yeterlidir.

### ⛔ Eğitim hata vermeden durdu ise

Colab'de en sık sebep **oturumun RAM yetersizliğinden öldürülmesidir**. Belirtisi
tipiktir: eğitim 1-2 epoch ilerler, sonra hücre sessizce durur — traceback yoktur,
çünkü süreç Python'a haber vermeden öldürülür.

Log'da şuna bakın:

```
train: Caching images (31.6GB RAM): 100%
val:   Caching images (5.3GB RAM): 100%
Using 12 dataloader workers
```

Bu iki satırın toplamı (~37 GB) **ana süreçte** tutulur; ayrıca her worker bu belleğin
bir kısmını kopyalar. 83 GB RAM'de bile sınır aşılabilir.

| Çözüm | Nasıl |
|-------|-------|
| Önbelleği kapat (en güvenli) | `OVERRIDES = {'cache': False}` |
| Diske önbellekle | `OVERRIDES = {'cache': 'disk'}` — RAM yerine disk kullanır |
| Çözünürlüğü düşür | `OVERRIDES = {'imgsz': 640}` — bellek ihtiyacı ~2.5 kat azalır |
| Worker azalt | `OVERRIDES = {'workers': 4}` |

5️⃣ hücresi artık bu kararı **görüntü sayısına göre** verir ve gerekçesini yazar; yine de
elle geçersiz kılmak isterseniz yukarıdaki `OVERRIDES` satırlarını kullanın.

> Not: `Box and segment counts should be equal` uyarısı **hata değildir** — bazı
> kaynaklarda poligon (segmentasyon) etiketi var, Ultralytics yalnızca kutuları kullanır.
> Nesne tespiti eğitimini etkilemez.

In [ ]:
# ÖN KOŞUL: Yeni Colab oturumunda paketler ve değişkenler sıfırlanır.
# Bu hücre tek başına çalışmaz; kısa yol: Çalışma zamanı → Öncekileri çalıştır (Run before)
_eksik = [a for a in ('TRAIN_CONFIG', 'DATA_YAML_PATH', 'MODELS_DIR', 'REPO_DIR', 'dataset_hazirla')
          if a not in globals()]
if _eksik:
    raise RuntimeError('Önce üstteki hücreleri çalıştırın — eksik: ' + ', '.join(_eksik) +
                       '\nColab menüsü: Çalışma zamanı → Öncekileri çalıştır (Run before)')

try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    raise RuntimeError('ultralytics kurulu değil — 1️⃣ Kurulum hücresini çalıştırın '
                       '(veya Çalışma zamanı → Öncekileri çalıştır).') from None

# Dataset hazır değilse otomatik hazırla (4️⃣ hücresini atlasanız da çalışır)
dataset_hazirla()

import time, shutil
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════
#  MOD SEÇİMİ  —  ne yapılacağını buradan belirleyin
#
#    'otomatik'  → Yarım kalmış eğitim varsa DEVAM eder, bitmişse ATLAR,
#                  hiç yoksa BAŞTAN başlatır.            (önerilen)
#    'devam'     → Yarım kalmış eğitimden DEVAM etmeye zorlar.
#                  Devam edilecek eğitim yoksa hata verir, sessizce
#                  sıfırdan başlatmaz.
#    'sifirdan'  → Mevcut checkpoint'leri YOK SAYAR, yepyeni bir koşu açar.
#                  OVERRIDES ile ayar değiştirdiyseniz bunu kullanın.
#    'ince_ayar'  → Mevcut MODELİNİZDEN (best.pt) devam eder: ağırlıklar
#                  devralınır, sıfırdan başlatılmaz. Veri setine yeni görüntü
#                  eklediğinizde kullanın — çok daha az epoch'ta yakınsar.
#                  'devam' ile KARIŞTIRMAYIN: 'devam' yarım kalan koşuyu
#                  sürdürür ve yeni veriyi/ayarı almaz.
# ═══════════════════════════════════════════════════════════════════════════
MOD = 'otomatik'

# 'ince_ayar' modunda başlangıç ağırlığı: boş bırakılırsa Drive'daki en yeni
# best_*.pt kullanılır. Belirli bir modeli istiyorsanız yolunu yazın.
INCE_AYAR_AGIRLIK = ''


# NOT: Sıfırdan ve ince ayar koşuları AYRI değerlendirilir.
# Adları da hedef epoch sayıları da farklıdır (örn. strawberry_exp / 200 ile
# strawberry_exp_ince_ayar / 60). Ayrılmazsa iki hata çıkar:
#   - 'strawberry_exp*' deseni ince ayar koşusunu da yakalar ve yanlış koşudan
#     devam edilir,
#   - 60 epochta biten bir ince ayar, 200 hedefiyle kıyaslanıp "yarım" sanılır.
INCE_EK = '_ince_ayar'

# Bu modelin kosularinin ad oneki. Kosu klasorleri results/ altinda ORTAK durur;
# onek olmadan organ egitimi baslatildiginda birlesik modelin 'strawberry_ince_ayar'
# kosusu "bitmis" sayilip egitim ATLANIYOR, dahasi o modelin agirligi organ.pt
# adiyla kopyalaniyordu. Birlesik kurulumda iki kosu ailesi vardir
# (strawberry_exp* ve strawberry_ince_ayar*); ortak onekleri 'strawberry'.
_secili = globals().get('EGITILECEK', 'birlesik')
KOSU_ONEKI = (_secili if _secili not in ('birlesik', '')
              else str(TRAIN_CONFIG['name']).split('_')[0])
print(f'🔖 Bu modelin koşu öneki: {KOSU_ONEKI!r}  (yalnızca bunlar dikkate alınır)')


def _ince_mi(d):
    return d.name.endswith(INCE_EK) or INCE_EK in d.name


def _bu_modelin(d):
    """Klasor SECILEN modele mi ait? Baska modelin kosusu asla karismaz."""
    return d.name.startswith(KOSU_ONEKI)


def _kosular(ince=None):
    """project/name* dizinlerini en yeniden eskiye sıralar.

    ince=True  → yalnızca ince ayar koşuları
    ince=False → yalnızca sıfırdan koşular
    ince=None  → hepsi
    """
    base = Path(TRAIN_CONFIG['project'])
    if not base.exists():
        return []
    # DİKKAT: İnce ayar koşusunun adı sıfırdan eğitiminkinden BAĞIMSIZDIR
    # (train_config: strawberry_exp, finetune_config: strawberry_ince_ayar).
    # Bu yüzden ince ayar koşuları AD EKİNDEN, sıfırdan koşular yapılandırmadaki
    # ad ön ekinden bulunur. Tek desene ('strawberry_exp*') güvenmek, farklı
    # adlandırılmış ince ayar koşusunu görünmez yapar ve devam edilemez.
    tumu = [p for p in base.iterdir() if p.is_dir() and _bu_modelin(p)]
    inceler = [p for p in tumu if _ince_mi(p)]
    sifirdanlar = [p for p in tumu
                   if not _ince_mi(p) and p.name.startswith(TRAIN_CONFIG['name'])]
    hepsi = inceler if ince is True else (sifirdanlar if ince is False
                                          else inceler + sifirdanlar)
    return sorted(hepsi, key=lambda p: p.stat().st_mtime, reverse=True)


def _hedef_epoch(d):
    """Koşunun kendi hedef epoch sayısı — args.yaml'dan okunur.

    Ultralytics her koşunun ayarlarını args.yaml'a yazar. Buradan okumak,
    yapılandırma sonradan değişse bile doğru karşılaştırma sağlar.
    """
    args = d / 'args.yaml'
    if args.exists():
        try:
            import yaml as _y
            v = (_y.safe_load(args.read_text(encoding='utf-8')) or {}).get('epochs')
            if v:
                return int(v)
        except Exception:
            pass
    return int(TRAIN_CONFIG['epochs'])


def _durum(d):
    """(durum, tamamlanan_epoch) → durum: 'yok' | 'yarim' | 'bitti'"""
    if not (d / 'weights' / 'last.pt').exists():
        return 'yok', 0
    csv = d / 'results.csv'
    n = 0
    if csv.exists():
        try:
            n = max(0, sum(1 for _ in csv.open(encoding='utf-8', errors='ignore')) - 1)
        except OSError:
            n = 0
    return ('bitti' if n >= _hedef_epoch(d) else 'yarim'), n


def _devam_edilebilir(ince=None):
    """Devam edilecek koşuyu döner: EN ÇOK İLERLEMİŞ yarım koşu.

    "En yeni" değil "en ilerlemiş" seçilir; aksi halde yanlışlıkla açılmış
    1 epoch'luk yeni bir koşu, 120 epoch ilerlemiş koşunun önüne geçerdi.
    """
    adaylar = [(d, _durum(d)[1]) for d in _kosular(ince) if _durum(d)[0] == 'yarim']
    if not adaylar:
        return None, 0
    return max(adaylar, key=lambda x: (x[1], x[0].stat().st_mtime))


# Mevcut koşuları göster — hangi eğitimin nerede kaldığı net görünsün
# Koşu adı seçilen modele göre ayrılır: farklı modeller birbirinin klasörünü
# ezmesin ve "yarım kalan" mantığı doğru koşuyu bulsun.
if 'EGITILECEK' in globals() and EGITILECEK not in ('birlesik', ''):
    TRAIN_CONFIG['name'] = EGITILECEK

kosular = _kosular()
if kosular:
    print(f'📋 Mevcut koşular ({TRAIN_CONFIG["project"]}):')
    for d in kosular:
        durum, n = _durum(d)
        etiket = {'bitti': '✅ tamamlandı', 'yarim': '⏸️ yarım kaldı', 'yok': '— checkpoint yok'}[durum]
        tur = '🔁 ince ayar' if _ince_mi(d) else '↻ sıfırdan  '
        isaret = ''
        if d == _devam_edilebilir(ince=_ince_mi(d))[0]:
            isaret = ('   ⬅ ince ayarda devam edilecek' if _ince_mi(d)
                      else '   ⬅ devam edilecek')
        print(f'   {d.name:<26} {tur}  {n:>4}/{_hedef_epoch(d)} epoch  {etiket}{isaret}')
    print()

# 'devam' ve 'otomatik' SIFIRDAN koşuları sürdürür; ince ayar koşusundan
# yanlışlıkla devam edilmesin diye ince=False verilir.
devam_dir, devam_epoch = _devam_edilebilir(ince=False)
model = None
t0 = time.time()

if MOD == 'devam':
    if not devam_dir:
        raise RuntimeError(
            "MOD='devam' seçildi ama devam edilebilecek yarım eğitim yok.\n"
            "Yukarıdaki listeye bakın; yeni bir eğitim başlatmak için MOD='sifirdan' yapın."
        )
    print(f'🔄 DEVAM: {devam_dir.name} ({devam_epoch}. epoch\'tan sonrası)')
    print("   (ayarlar checkpoint'ten okunur; OVERRIDES bu modda etkisizdir)")
    model = YOLO(str(devam_dir / 'weights' / 'last.pt'))
    model.train(resume=True)

elif MOD == 'sifirdan':
    print('↻ SIFIRDAN: mevcut checkpoint\'ler yok sayılıyor, yeni koşu açılıyor')
    model = YOLO(TRAIN_CONFIG['model'])
    model.train(data=DATA_YAML_PATH, **TRAIN_CONFIG)

elif MOD == 'otomatik':
    bitmis = [d for d in kosular if _durum(d)[0] == 'bitti']
    if devam_dir:
        print(f'🔄 Yarım kalmış eğitim bulundu → devam ediliyor: {devam_dir.name} '
              f'({devam_epoch}. epoch\'tan sonrası)')
        print("   (ayarlar checkpoint'ten okunur; OVERRIDES bu modda etkisizdir)")
        model = YOLO(str(devam_dir / 'weights' / 'last.pt'))
        try:
            model.train(resume=True)
        except Exception as e:
            print(f'⚠️ Devam edilemedi ({type(e).__name__}: {e}) → yeni koşu başlatılıyor')
            model = YOLO(TRAIN_CONFIG['model'])
            model.train(data=DATA_YAML_PATH, **TRAIN_CONFIG)
    elif bitmis:
        RUN_DIR = bitmis[0]
        print(f'✅ {KOSU_ONEKI} eğitimi zaten tamamlanmış: {RUN_DIR.name}')
        print('   Değerlendirme hücresine geçebilirsiniz.')
        print("   Yeniden eğitmek için: MOD = 'sifirdan'")
    else:
        print('🚀 Eğitim başlıyor (ilk koşu)...')
        model = YOLO(TRAIN_CONFIG['model'])   # yolo26s.pt ilk çalıştırmada indirilir
        model.train(data=DATA_YAML_PATH, **TRAIN_CONFIG)

elif MOD == 'ince_ayar':
    # --- Yarım kalmış ince ayar var mı? ----------------------------------
    # Colab oturumu kopabilir (RAM, zaman aşımı, tarayıcı). Yeni koşu açmak
    # yerine kaldığı yerden devam edilir: Ultralytics resume, optimizer
    # durumunu ve öğrenme oranı takvimini checkpoint'ten okur — baştan
    # başlamak hem süreyi hem takvimi boşa harcardı.
    _yarim_dir, _yarim_epoch = _devam_edilebilir(ince=True)
    if _yarim_dir is not None:
        print(f'🔄 YARIM KALMIŞ İNCE AYAR BULUNDU: {_yarim_dir.name}')
        print(f'   {_yarim_epoch}/{_hedef_epoch(_yarim_dir)} epoch tamamlanmış '
              '→ kaldığı yerden devam ediliyor')
        print("   (ayarlar checkpoint'ten okunur; OVERRIDES bu modda etkisizdir)")
        print('   Baştan başlatmak isterseniz o klasörü silin veya '
              "MOD = 'sifirdan' kullanın.")
        model = YOLO(str(_yarim_dir / 'weights' / 'last.pt'))
        try:
            model.train(resume=True)
        except Exception as _e:
            print(f'⚠️ Devam edilemedi ({type(_e).__name__}: {_e})')
            print('   Yeni bir ince ayar koşusu açılıyor...')
            _yarim_dir = None
    if _yarim_dir is None:
        _ince_ayar_yeni_kosu = True
    else:
        _ince_ayar_yeni_kosu = False

if MOD == 'ince_ayar' and _ince_ayar_yeni_kosu:
    # --- Başlangıç ağırlığını bul ---------------------------------------
    if INCE_AYAR_AGIRLIK:
        agirlik = Path(INCE_AYAR_AGIRLIK)
    else:
        adaylar = sorted(MODELS_DIR.glob('best*.pt'),
                         key=lambda p: p.stat().st_mtime, reverse=True)
        if not adaylar:
            raise RuntimeError(
                f"İnce ayar için başlangıç ağırlığı bulunamadı: "
                f"{MODELS_DIR}/best*.pt" + chr(10) +
                "Önce bir eğitim tamamlayın (MOD='sifirdan') ya da INCE_AYAR_AGIRLIK "
                "değişkenine model yolunu yazın.")
        agirlik = adaylar[0]
    if not agirlik.exists():
        raise RuntimeError(f'Başlangıç ağırlığı yok: {agirlik}')

    # --- SINIF UYUMU: eğitim başlamadan kontrol edilir -------------------
    # Sınıf sayısı/sırası uyuşmazsa Ultralytics hata vermez; tespit başını
    # sessizce yeniden kurar veya ID kaydığı için yanlış sınıfları öğrenir.
    # Sonuç ancak saatler sonra fark edilir — bu yüzden burada durduruyoruz.
    import torch, yaml as _yaml
    _ck = torch.load(str(agirlik), map_location='cpu', weights_only=False)
    _m = _ck.get('model') or _ck.get('ema')
    _ad = getattr(_m, 'names', None) or _ck.get('names') or {}
    _agirlik_siniflari = [_ad[i] for i in sorted(_ad)] if isinstance(_ad, dict) else list(_ad)
    _veri = _yaml.safe_load(Path(DATA_YAML_PATH).read_text(encoding='utf-8')) or {}
    _n = _veri.get('names', {})
    _dataset_siniflari = [_n[i] for i in sorted(_n)] if isinstance(_n, dict) else list(_n)

    if _agirlik_siniflari != _dataset_siniflari:
        mesaj = [
            '', '=' * 72,
            '⛔ EĞİTİM BAŞLATILMADI — sınıf listeleri uyuşmuyor',
            '=' * 72,
            f'Başlangıç ağırlığı : {agirlik}',
            f'  {len(_agirlik_siniflari)} sınıf: {_agirlik_siniflari}',
            f'Dataset            : {DATA_YAML_PATH}',
            f'  {len(_dataset_siniflari)} sınıf: {_dataset_siniflari}', '']
        if len(_agirlik_siniflari) != len(_dataset_siniflari):
            _yeni = [c for c in _dataset_siniflari if c not in _agirlik_siniflari]
            mesaj.append(f'Sınıf SAYISI farklı. Datasette olup ağırlıkta olmayan: {_yeni}')
        else:
            for _i, (_a, _b) in enumerate(zip(_agirlik_siniflari, _dataset_siniflari)):
                if _a != _b:
                    mesaj.append(f'ID {_i}: ağırlıkta "{_a}", datasette "{_b}" — sıra kaymış.')
        mesaj += ['', 'NE YAPMALI?',
                  "  • Yeni SINIF eklediyseniz ince ayar yapılamaz (tespit başı yeniden",
                  "    kurulmalı): MOD = 'sifirdan' ile eğitin.",
                  "  • Sıra kaydıysa configs/urunler/<urun>/siniflar.yaml içindeki ID",
                  "    haline getirin — ID bir kez verilir, değiştirilmez.",
                  "  • Yanlış model seçildiyse INCE_AYAR_AGIRLIK değişkenini düzeltin.",
                  '=' * 72]
        raise RuntimeError(chr(10).join(mesaj))

    print(f'🔁 İNCE AYAR: {agirlik.name}')
    print(f'   {len(_dataset_siniflari)} sınıf uyumlu ✓ — ağırlıklar devralınıyor')

    # --- İnce ayar YAPILANDIRMASI ---------------------------------------
    # ÖNEMLİ: TRAIN_CONFIG sıfırdan eğitim ayarlarıdır (epochs 200,
    # optimizer auto). optimizer 'auto' lr0'ı YOK SAYAR ve sıfırdan eğitime
    # uygun yüksek bir öğrenme oranı seçer — ince ayarda bu, öğrenilmiş
    # ağırlıkları bozar. Bu yüzden finetune_config.yaml yüklenir.
    import yaml as _yml
    _ince_yol = Path(REPO_DIR) / 'configs' / 'finetune_config.yaml'
    if _ince_yol.exists():
        _ince = _yml.safe_load(_ince_yol.read_text(encoding='utf-8')) or {}
        # GPU'ya göre otomatik seçilenler korunur (bu makineye özel)
        # 'project' ZORUNLU: yapilandirmadaki 'runs/train' goreli bir yoldur ve
        # Colab'in GECICI diskine yazar; oturum kopunca checkpoint'ler ve
        # results.csv kaybolur, egitime devam edilemez. TRAIN_CONFIG'te bu
        # deger Drive'daki RESULTS_DIR ile degistirilmistir.
        for _k in ('batch', 'workers', 'cache', 'amp', 'device', 'project'):
            if _k in TRAIN_CONFIG:
                _ince[_k] = TRAIN_CONFIG[_k]
        try:
            _ince.update(OVERRIDES)          # kullanıcının elle ayarları
        except NameError:
            pass
        print(f'   ⚙️ Ayarlar: configs/finetune_config.yaml')
    else:
        _ince = dict(TRAIN_CONFIG)
        _ince['epochs'] = min(int(_ince.get('epochs', 200)), 70)
        _ince['optimizer'] = 'AdamW'
        _ince['lr0'] = 0.0008
        print('   ⚠️ finetune_config.yaml bulunamadı → güvenli ince ayar')
        print('      değerleri uygulandı (epochs 70, AdamW, lr0 0.0008).')

    _ince.pop('model', None)                       # ağırlık aşağıda veriliyor
    _ince.pop('sinif_kontrolu_atla', None)         # Ultralytics parametresi değil
    # Kosu adi SECILEN modelden turetilir. finetune_config.yaml'daki sabit ad
    # ('strawberry_ince_ayar') kullanilsaydi organ modelinin ince ayari birlesik
    # modelin klasorune yazar, ikisi birbirini ezerdi.
    if globals().get('EGITILECEK', 'birlesik') not in ('birlesik', ''):
        _ince['name'] = f'{EGITILECEK}{INCE_EK}'
    elif not str(_ince.get('name', '')).endswith('ince_ayar'):
        _ince['name'] = str(_ince.get('name', 'strawberry_exp')) + INCE_EK
    print(f'   epochs={_ince.get("epochs")}  optimizer={_ince.get("optimizer")}  '
          f'lr0={_ince.get("lr0")}  imgsz={_ince.get("imgsz")}  batch={_ince.get("batch")}')
    print(f'   koşu adı: {_ince.get("name")}')
    print('   kayıt yeri: {}/{}'.format(_ince.get("project"), _ince.get("name")))
    print('   checkpoint: her {} epochta kaydedilir'.format(_ince.get("save_period", 10)))
    model = YOLO(str(agirlik))
    model.train(data=DATA_YAML_PATH, **_ince)

if MOD not in ('otomatik', 'devam', 'sifirdan', 'ince_ayar'):
    raise ValueError(f"MOD geçersiz: {MOD!r} — 'otomatik', 'devam', 'sifirdan' "
                     "veya 'ince_ayar' olmalı")

if model is not None:
    print(f'\n✅ Eğitim bitti: {(time.time()-t0)/3600:.2f} saat')
    # Çıktı dizinini VARSAYMA, trainer'dan al
    RUN_DIR = Path(model.trainer.save_dir)

print("📊 Sonuçlar:", RUN_DIR)
print("   Drive'da tutulur; Colab oturumu kapansa da silinmez.")
best_path = RUN_DIR / "weights" / "best.pt"

# Hiyerarşik mimaride her uzman modelin boru hattında beklenen bir DOSYA ADI
# vardır (configs/urunler/<urun>/modeller.yaml). Eğitim çıktısı hep "best.pt"
# olduğu için elle adlandırmak gerekiyordu; bu üç sessiz hata kaynağıdır:
# yanlış ada kopyalama (model hiç kullanılmaz), yanlış modeli kopyalama,
# sınıfları uymayan model. Burada doğru adla da kopyalanır.
BORU_HATTI_ADLARI = {
    "organ_detection": "organ.pt",
    "leaf_disease": "leaf_disease.pt",
    "fruit_disease": "fruit_disease.pt",
    "fruit_ripeness": "fruit_ripeness.pt",
    "bocek_teshis": "bocek_teshis.pt",       # ayri akis, ROI boru hattina girmez
    "pest_detection": "pest_detection.pt",
    "birlesik": "best.pt",
}

if best_path.exists():
    dest = MODELS_DIR / f"best_{RUN_DIR.name}.pt"
    shutil.copy(best_path, dest)
    print("🏆 En iyi model Drive'a kopyalandı:", dest)

    _ad = globals().get("EGITILECEK", "birlesik")
    _urun = globals().get("URUN", "cilek")
    _dosya = BORU_HATTI_ADLARI.get(_ad)

    # Kopyalamadan ONCE: agirligin siniflari bu modelin dataset'iyle ayni mi?
    # Egitim atlandiginda RUN_DIR baska bir kosuyu gosterebilir; dogrulama
    # olmadan yanlis model boru hatti adiyla kopyalanir ve sistem sessizce
    # yanlis calisir. model_kur.py ayni kontrolu kurulumda tekrar yapar.
    import yaml as _yk, torch as _tk
    try:
        _ck = _tk.load(str(best_path), map_location='cpu', weights_only=False)
        _mm = _ck.get('model') or _ck.get('ema')
        _nm = getattr(_mm, 'names', None) or _ck.get('names') or {}
        _agirlik_sin = [_nm[i] for i in sorted(_nm)] if isinstance(_nm, dict) else list(_nm)
    except Exception as _e:
        print(f'⚠️ Ağırlığın sınıfları okunamadı ({type(_e).__name__}) — kopyalama atlandı.')
        _agirlik_sin = None

    _vd = _yk.safe_load(Path(DATA_YAML_PATH).read_text(encoding='utf-8')) or {}
    _vn = _vd.get('names', {})
    _veri_sin = [_vn[i] for i in sorted(_vn)] if isinstance(_vn, dict) else list(_vn)

    if _dosya and _agirlik_sin is not None and _agirlik_sin != _veri_sin:
        print('')
        print('=' * 72)
        print('⛔ BORU HATTI ADIYLA KOPYALANMADI — sınıflar uyuşmuyor')
        print('=' * 72)
        print(f'  Koşu       : {RUN_DIR.name}')
        print(f'  Ağırlıkta  : {len(_agirlik_sin)} sınıf {_agirlik_sin}')
        print(f'  Datasette  : {len(_veri_sin)} sınıf {_veri_sin}')
        print('')
        print(f'  Bu koşu {_ad} modeline ait DEĞİL. Muhtemel sebep: eğitim')
        print('  atlandı ve RUN_DIR başka bir koşuyu gösteriyor.')
        print("  Çözüm: MOD = 'sifirdan' ile bu modeli gerçekten eğitin.")
        print('=' * 72)
        _dosya = None

    if _dosya:
        _urun_kls = MODELS_DIR / _urun
        _urun_kls.mkdir(parents=True, exist_ok=True)
        _hedef = _urun_kls / _dosya
        shutil.copy(best_path, _hedef)
        print("📦 Boru hattı adıyla da kopyalandı:", _hedef)
        print("")
        print("   Kullanmak için bu dosyayı projede şu konuma indirin:")
        print(f"      models/{_urun}/{_dosya}")
        print("   Doğrulayarak kurmak için (sınıfları kütükle karşılaştırır):")
        print("      python scripts/model_kur.py <kutuk_adi> <indirdiginiz.pt>")
else:
    print("⚠️ best.pt yok — eğitim loglarını kontrol edin.")


## 7️⃣ Değerlendirme — sınıf bazlı (ticari karar buradan verilir)

Genel mAP tek başına yanıltıcıdır: ortalama iyi görünürken tek bir hastalıkta recall
çok düşük olabilir. Az örnekli sınıflara (Anthracnose Fruit Rot, Powdery Mildew Fruit)
ayrıca bakın.

In [ ]:
# ÖN KOŞUL: Yeni Colab oturumunda paketler ve değişkenler sıfırlanır.
# Bu hücre tek başına çalışmaz; kısa yol: Çalışma zamanı → Öncekileri çalıştır (Run before)
_eksik = [a for a in ('DATA_YAML_PATH', 'find_run_dir', 'dataset_hazirla') if a not in globals()]
if _eksik:
    raise RuntimeError('Önce üstteki hücreleri çalıştırın — eksik: ' + ', '.join(_eksik) +
                       '\nColab menüsü: Çalışma zamanı → Öncekileri çalıştır (Run before)')

try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    raise RuntimeError('ultralytics kurulu değil — 1️⃣ Kurulum hücresini çalıştırın '
                       '(veya Çalışma zamanı → Öncekileri çalıştır).') from None

# Dataset hazır değilse otomatik hazırla (4️⃣ hücresini atlasanız da çalışır)
dataset_hazirla()

from pathlib import Path

# Koşu dizini: eğitim bu oturumda yapıldıysa gerçek save_dir, değilse en yeni koşu
run_dir = RUN_DIR if 'RUN_DIR' in globals() else find_run_dir()
best_path = run_dir / 'weights' / 'best.pt'
print('📂 Koşu dizini:', run_dir)

if not best_path.exists():
    print('⚠️ best.pt bulunamadı — önce eğitim hücresini çalıştırın.')
else:
    model = YOLO(str(best_path))
    m = model.val(data=DATA_YAML_PATH)   # eğitimdekiyle AYNI config

    print('\n' + '='*58)
    print(f'GENEL  mAP50: {m.box.map50:.4f} | mAP50-95: {m.box.map:.4f} | '
          f'P: {m.box.mp:.4f} | R: {m.box.mr:.4f}')
    print('='*58)
    print(f"\n{'Sınıf':<24}{'P':>8}{'R':>8}{'mAP50':>9}")
    cls_names = model.names
    zayif = []
    for i, c in enumerate(m.box.ap_class_index):
        r = float(m.box.r[i])
        flag = '  ⚠️ düşük recall' if r < 0.75 else ''
        if r < 0.75:
            zayif.append(cls_names[int(c)])
        print(f'{cls_names[int(c)]:<24}{m.box.p[i]:>8.3f}{r:>8.3f}{m.box.ap50[i]:>9.3f}{flag}')

    if zayif:
        print(f'\n⚠️ Recall < 0.75 olan sınıflar: {", ".join(zayif)}')
        print('   → Bu sınıflar için augmentasyon çarpanını artırmak yerine GERÇEK veri toplayın.')
    print('\n💡 confusion_matrix.png: hangi hastalık hangisiyle karışıyor?')

## 7️⃣.1 🔁 Eski model ile karşılaştırma (ince ayar sonrası)

İnce ayar yaptıysanız **gerçekten iyileşti mi?** Bunu ölçmenin tek geçerli yolu,
iki modeli de **aynı test setinde** çalıştırmaktır.

> ⚠️ Eski modelin eğitim sonundaki mAP'ı ile yeninin mAP'ını kıyaslamak **yanlıştır**:
> veri seti değiştiyse (yeni kaynak eklendi, etiketler düzeltildi) iki sayı farklı
> ölçütlerden gelir.

Rapor **sınıf bazındadır**: toplam mAP artarken tek tek sınıflar gerileyebilir —
özellikle yeni veri belirli sınıflara yoğunlaşırsa (catastrophic forgetting).
Ortalama bunu gizler, tablo gizlemez.

Sıfırdan eğitim yaptıysanız bu hücreyi atlayabilirsiniz (karşılaştırılacak önceki
model yoksa kendisi atlar).

In [ ]:
# ÖN KOŞUL: Yeni Colab oturumunda paketler ve değişkenler sıfırlanır.
# Bu hücre tek başına çalışmaz; kısa yol: Çalışma zamanı → Öncekileri çalıştır (Run before)
_eksik = [a for a in ("DATA_YAML_PATH", "MODELS_DIR", "REPO_DIR", "find_run_dir")
          if a not in globals()]
if _eksik:
    raise RuntimeError("Önce üstteki hücreleri çalıştırın — eksik: " + ", ".join(_eksik) +
                       chr(10) + "Colab menüsü: Çalışma zamanı → Öncekileri çalıştır")

import sys
from pathlib import Path

# Karşılaştırma mantığı depodaki scriptte durur; notebook her çalıştırmada
# "git pull" yaptığı için düzeltmeler buraya kendiliğinden yansır.
sys.path.insert(0, str(Path(REPO_DIR) / "scripts"))
import importlib
import model_karsilastir
importlib.reload(model_karsilastir)      # açık oturumda eski sürüm kalmasın

# --- YENİ model: bu oturumda eğitilen koşu -------------------------------
run_dir = RUN_DIR if "RUN_DIR" in globals() else find_run_dir()
yeni_model = Path(run_dir) / "weights" / "best.pt"

# --- ESKİ model: ince ayarın başladığı ağırlık ---------------------------
# Öncelik: bu oturumda ince ayar yapıldıysa onun başlangıç ağırlığı.
# Yoksa Drive'daki, yeni koşuya ait OLMAYAN en yeni best*.pt.
if "agirlik" in globals() and Path(agirlik).exists():
    eski_model = Path(agirlik)
else:
    adaylar = [q for q in sorted(Path(MODELS_DIR).glob("best*.pt"),
                                 key=lambda q: q.stat().st_mtime, reverse=True)
               if Path(run_dir).name not in q.name]
    eski_model = adaylar[0] if adaylar else None

# İsterseniz elle belirtin:
# eski_model = Path(MODELS_DIR) / "best_strawberry_exp-3.pt"

SPLIT = "test"      # test önerilir: val eğitim sırasında model seçimi için kullanıldı

print("Eski :", eski_model)
print("Yeni :", yeni_model)
print()

if not yeni_model.exists():
    print("⚠️ Yeni model yok — önce eğitim hücresini çalıştırın.")
elif eski_model is None:
    print("ℹ️ Karşılaştırılacak önceki model yok (ilk eğitim).")
    print("   Bu normaldir: ince ayar yaptıktan sonra bu hücre anlamlı olur.")
elif Path(eski_model).resolve() == yeni_model.resolve():
    print("ℹ️ Eski ve yeni model aynı dosya — karşılaştırılacak bir şey yok.")
else:
    _imgsz = TRAIN_CONFIG.get("imgsz", 1024) if "TRAIN_CONFIG" in globals() else 1024
    _batch = TRAIN_CONFIG.get("batch", 8) if "TRAIN_CONFIG" in globals() else 8

    eski_sonuc = model_karsilastir.olc(str(eski_model), DATA_YAML_PATH, SPLIT, _imgsz, _batch)
    yeni_sonuc = model_karsilastir.olc(str(yeni_model), DATA_YAML_PATH, SPLIT, _imgsz, _batch)
    KARSILASTIRMA = model_karsilastir.rapor(eski_sonuc, yeni_sonuc)

    # Raporu Drive'a yaz — oturum kapansa da karar kaydı kalsın
    import json as _json
    _rapor = Path(MODELS_DIR) / ("karsilastirma_" + Path(run_dir).name + ".json")
    _rapor.write_text(_json.dumps(KARSILASTIRMA, ensure_ascii=False, indent=2),
                      encoding="utf-8")
    print(chr(10) + "📝 Rapor kaydedildi:", _rapor)

    # Karar: yalnızca genel iyileşme VAR ve hiçbir sınıf gerilemediyse otomatik kopyala
    if KARSILASTIRMA["genel_fark"] > 0.005 and not KARSILASTIRMA["gerileyen"]:
        import shutil
        _hedef = Path(MODELS_DIR) / "best.pt"
        shutil.copy(yeni_model, _hedef)
        print("🏆 Yeni model daha iyi. Drive kopyası:", _hedef)
        print("   Uygulamada kullanmak için bu dosyayı models/best.pt yapın.")
    else:
        print("⏸️ Otomatik kopyalama yapılmadı — yukarıdaki karar satırını okuyun.")


In [ ]:
# Eğitim grafikleri ve confusion matrix
from IPython.display import Image, display
from pathlib import Path

rd = RUN_DIR if 'RUN_DIR' in globals() else find_run_dir()
print('📂 Koşu dizini:', rd)
for f in ('results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg'):
    p = rd / f
    if p.exists():
        print(f)
        display(Image(filename=str(p)))

## 8️⃣ Örnek tahminler

Yüksek çözünürlüklü saha fotoğraflarında küçük lezyonlar için
`scripts/sahi_predict.py` (dilimli inference) kullanın.

In [ ]:
# ÖN KOŞUL: Yeni Colab oturumunda paketler ve değişkenler sıfırlanır.
# Bu hücre tek başına çalışmaz; kısa yol: Çalışma zamanı → Öncekileri çalıştır (Run before)
_eksik = [a for a in ('DATA_YAML_PATH', 'find_run_dir', 'dataset_hazirla') if a not in globals()]
if _eksik:
    raise RuntimeError('Önce üstteki hücreleri çalıştırın — eksik: ' + ', '.join(_eksik) +
                       '\nColab menüsü: Çalışma zamanı → Öncekileri çalıştır (Run before)')

try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    raise RuntimeError('ultralytics kurulu değil — 1️⃣ Kurulum hücresini çalıştırın '
                       '(veya Çalışma zamanı → Öncekileri çalıştır).') from None

# Dataset hazır değilse otomatik hazırla (4️⃣ hücresini atlasanız da çalışır)
dataset_hazirla()

import cv2, yaml
import matplotlib.pyplot as plt
from pathlib import Path

cfg = yaml.safe_load(Path(DATA_YAML_PATH).read_text(encoding='utf-8'))
root = Path(cfg.get('path') or Path(DATA_YAML_PATH).parent)
val_dirs = cfg['val'] if isinstance(cfg['val'], list) else [cfg['val']]

imgs = []
for d in val_dirs:
    p = (root / d).resolve()
    if p.exists():
        imgs += sorted(p.glob('*.jpg'))[:2]

best_path = (RUN_DIR if 'RUN_DIR' in globals() else find_run_dir()) / 'weights' / 'best.pt'
if imgs and best_path.exists():
    model = YOLO(str(best_path))
    for ip in imgs[:5]:
        r = model(str(ip), verbose=False)[0]
        plt.figure(figsize=(11, 7))
        plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off')
        plt.title(ip.name); plt.show()
        counts = {}
        for b in r.boxes:
            n = model.names[int(b.cls[0])]
            counts[n] = counts.get(n, 0) + 1
        print(f"{ip.name}: {len(r.boxes)} tespit → {counts or 'yok'}\n" + '-'*50)
else:
    print('⚠️ Görüntü veya model bulunamadı.')

---
## 📝 Notlar

**Sonuçlar nerede?** Eğitim doğrudan **Drive'a** yazar — oturum kapansa da kalır:
```
MyDrive/SmartFarmStrawberryDisease/
├── results/<EGITILECEK>/          # grafikler, confusion matrix, weights/best.pt
└── best_models/cilek/<boru_hatti_adi>.pt
```
Boru hattı adları: `organ.pt` · `yaprak_hastalik.pt` · `meyve_hastalik.pt` ·
`olgunluk.pt` · `zararli.pt`. İndirdikten sonra projede:
```bash
python scripts/model_kur.py organ <indirdiginiz.pt>   # sınıfları doğrulayıp kurar
```

**Colab Pro+ ipuçları**

| Ayar | Nerede | Etkisi |
|---|---|---|
| **A100 GPU** | Runtime → Change runtime type | En hızlı; 200 epoch ~3-4 saat (T4'te 15+ saat) |
| **High-RAM** | Runtime → Change runtime type | A100'de `cache='ram'` açılır, disk darboğazı kalkar |
| **Background execution** | Runtime menüsü | Tarayıcı kapalıyken de eğitim sürer |

GPU tipi **koddan seçilemez** — runtime ayarıdır. Kod, hangi GPU verildiyse
`batch`/`workers`/`cache` değerlerini ona göre otomatik ayarlar.

**Sık karşılaşılan sorunlar**

| Sorun | Çözüm |
|---|---|
| `configs/... yok!` ya da eski hücre kodu | Sekme bayat: sekmeyi kapatın, linki tekrar açın, **Ctrl+Shift+R** |
| `images not found` | `data=` mutlak yol mu? (bu notebook otomatik yapar) |
| CUDA out of memory | 5️⃣ hücresinde `OVERRIDES = {'batch': <yarısı>}` |
| RAM doldu / oturum çöktü | `OVERRIDES = {'cache': False}` (High-RAM kapalıysa) |
| `numpy.dtype size changed` / cv2 import hatası | Runtime → Restart session, sonra 1️⃣ hücresi |
| `yolo26s.pt` yüklenemiyor | `!pip install -U ultralytics` (>=8.3.200 gerekir) |
| Oturum koptu | 6️⃣ bölümündeki "devam et" hücresi |

**Dataset güncellenirse:** Yerelde `python scripts/dataset_ayir.py --paketle` çalıştırıp
üretilen `datasets/cilek/<model>.zip` dosyasını Drive'da
`SmartFarmStrawberryDisease/datasets/cilek/` altındaki eskisinin üzerine yazın;
Colab'da o modelin `dataset/` klasörünü silip 4️⃣ hücresini tekrar çalıştırın.

**Başka bir model eğitmek:** 4️⃣ hücresinde `EGITILECEK` değerini değiştirip
Runtime → Run all. Her modelin kendi dataset zip'i, kendi koşu dizini ve kendi
`.pt` dosyası olur — birbirlerini ezmezler.